In [0]:
%sql
-- contains historial info and current info. In this case we dont include all the historial info, only stay with current data where end_dt = null 


In [0]:
%sql
-- check the uniqueness 
SELECT prd_key, COUNT(*)
FROM 
( 
  SELECT  
prd_id,
prd_key, 
prd_nm, 
cat_id,
pc.cat,
pc.subcat,
prd_cost,
prd_line,
prd_start_dt,
pc.maintenance
FROM silver.crm_prd_info pn 
LEFT JOIN silver.erp_px_cat_g1v2 pc  
ON pn.cat_id = pc.ID 
WHERE prd_end_dt IS NULL
) group by prd_key 
HAVING COUNT(*) > 1
-- result: no duplicate in prd_key 

In [0]:
%sql
-- Check if we have any duplicate cst_key after join
SELECT cst_id, COUNT(*) FROM 
(
SELECT ci.cst_id,
       ci.cst_key,
       ci.cst_firstname,
       ci.cst_lastname,
       ci.cst_marital_status,
       ci.cst_gndr,
       ci.cst_create_date,
       ca.bdate,
       ca.gen
FROM silver.crm_cust_info ci 
LEFT JOIN silver.erp_cust_az12 ca 
ON ci.cst_key = ca.CID 
LEFT JOIN silver.erp_loc_a101 la  
ON ci.cst_key = la.CID 
)
GROUP BY cst_id 
HAVING COUNT(*) > 1 



In [0]:
%sql
-- Check 2 gender column
SELECT DISTINCT ci.cst_gndr, ca.gen
FROM silver.crm_cust_info ci
LEFT JOIN silver.erp_cust_az12 ca 
ON ci.cst_key = ca.CID
LEFT JOIN silver.erp_loc_a101 la   
ON ci.cst_key = la.cid 
ORDER BY 1,2 
-- if the information of gender from 2 table different => ask expert which one is the master table. In this case, crm_cust_info is the master table and we will use information from this table 

In [0]:
%sql
-- check gender column 
SELECT DISTINCT ci.cst_gndr, ca.gen,
CASE WHEN cst_gndr != 'n/a' THEN cst_gndr  
     ELSE COALESCE(gen, 'n/a')
END AS new_gen
FROM silver.crm_cust_info ci
LEFT JOIN silver.erp_cust_az12 ca
ON ci.cst_key = ca.cid 
LEFT JOIN silver.erp_loc_a101 la
ON ci.cst_key = la.cid 
 

In [0]:
%sql
-- check if all dimension table can sucessfully join to the fact table 
SELECT * FROM gold.fact_sales f 
LEFT JOIN gold.dim_customer  c
ON f.customer_key = c.customer_key
LEFT JOIN gold.dim_product p 
ON f.product_key = p.product_key
WHERE f.product_key IS NULL